# CNN & LSTM All Tasks — Baseline

This notebook trains CNN and LSTM task models using the final hyperparameters copied from the fair CNN/LSTM fine-tuning notebook.

In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing 'src'.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)


PROJECT_ROOT = /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project


In [5]:
import pandas as pd
import torch

from src.data_prep import prepare_uji_data
from src.models.mlp_coordinates import CoordinateMLPModel
from src.models.mlp_joint import JointMLPModel
from src.models.mlp_multitask import MultiTaskMLPModel
from src.models.cnn_coordinates import CNNCoordinateModel
from src.models.cnn_joint import CNNJointModel
from src.models.cnn_multitask import CNNMultiTaskModel
from src.models.lstm_coordinates import LSTMCoordinateModel
from src.models.lstm_joint import LSTMJointModel
from src.models.lstm_multitask import LSTMMultiTaskModel
from src.training import TrainConfig, train_from_tensors


In [6]:
bundle = prepare_uji_data()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
in_dim = bundle.X_train.shape[1]

joint_y_train, joint_y_val = bundle.get_targets(["joint"])
mt_y_train, mt_y_val = bundle.get_targets(["building", "floor"])
coord_y_train, coord_y_val = bundle.get_targets(["longitude", "latitude"])

print("device:", device)
print("X train/val:", bundle.X_train.shape, bundle.X_val.shape)
print("coordinate_std:", bundle.coordinate_std)


device: cuda
X train/val: (19937, 1040) (1111, 1040)
coordinate_std: [123.39891  66.94215]


## Paste final fine-tuned settings

Copy the `FINAL_TUNED_CFGS` dictionary from the matching fine-tuning notebook and paste it into the cell below.

This is intentionally manual. It makes the baseline notebook self-contained and avoids silently depending on CSV files that may be missing, stale, or outside the submitted repo.


In [7]:
MANUAL_CONFIG_FAMILIES = ("cnn", "lstm")

# Paste the FINAL_TUNED_CFGS dictionary printed by the matching fine-tuning notebook here.
#
# Expected shape:
# FINAL_TUNED_CFGS = {
#     "mlp": {  # or "cnn" / "lstm"
#         "joint": {"lr": ..., "weight_decay": ..., "grad_clip_norm": ..., "max_epochs": ..., "patience": ..., "print_every": 5, "batch_size": 256, "val_batch_size": 512},
#         "multitask": {...},
#         "coordinate": {...},
#     }
# }

FINAL_TUNED_CFGS ={
    'cnn': {'coordinate': {'lr': 0.0005,
   'weight_decay': 0.0001,
   'grad_clip_norm': None,
   'max_epochs': 60,
   'patience': 12,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512},
  'joint': {'lr': 0.001,
   'weight_decay': 0.0005,
   'grad_clip_norm': None,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512},
  'multitask': {'lr': 0.001,
   'weight_decay': 0.0001,
   'grad_clip_norm': None,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512}},
 'lstm': {'coordinate': {'lr': 0.001,
   'weight_decay': 0.0001,
   'grad_clip_norm': 1.0,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512},
  'joint': {'lr': 0.0005,
   'weight_decay': 0.0005,
   'grad_clip_norm': 1.0,
   'max_epochs': 80,
   'patience': 15,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512},
  'multitask': {'lr': 0.0005,
   'weight_decay': 0.0001,
   'grad_clip_norm': 1.0,
   'max_epochs': 60,
   'patience': 12,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512}}
 }

TASKS = ("joint", "multitask", "coordinate")
TRAIN_KEYS = ("lr", "weight_decay", "max_epochs", "patience", "print_every", "batch_size", "val_batch_size", "grad_clip_norm")

def validate_manual_training_configs(final_cfgs: dict, families: tuple[str, ...], tasks: tuple[str, ...] = TASKS) -> pd.DataFrame:
    if not final_cfgs:
        raise RuntimeError(
            "FINAL_TUNED_CFGS is empty. Copy the final dictionary from the matching fine-tuning notebook first."
        )

    missing = []
    rows = []
    for family in families:
        if family not in final_cfgs:
            missing.append((family, "<family missing>"))
            continue
        for task in tasks:
            if task not in final_cfgs[family]:
                missing.append((family, task))
                continue
            spec = dict(final_cfgs[family][task])
            missing_keys = [k for k in ("lr", "weight_decay", "max_epochs", "patience") if k not in spec]
            if missing_keys:
                raise RuntimeError(f"Config for {(family, task)} is missing keys: {missing_keys}")
            rows.append({"family": family, "task": task, **{k: spec.get(k) for k in TRAIN_KEYS}})

    if missing:
        raise RuntimeError(f"Missing required tuned configs: {missing}")

    return pd.DataFrame(rows).sort_values(["family", "task"]).reset_index(drop=True)

# Change the families tuple only if this notebook is intentionally repurposed.
selected_cfg_df = validate_manual_training_configs(FINAL_TUNED_CFGS, families=MANUAL_CONFIG_FAMILIES)
selected_cfg_df


,family,task,lr,weight_decay,max_epochs,patience,print_every,batch_size,val_batch_size,grad_clip_norm
0,cnn,coordinate,0.0005,0.0001,60,12,5,256,512,NaN
1,cnn,joint,0.0010,0.0005,50,10,5,256,512,NaN
2,cnn,multitask,0.0010,0.0001,50,10,5,256,512,NaN
3,lstm,coordinate,0.0010,0.0001,50,10,5,256,512,1.0
4,lstm,joint,0.0005,0.0005,80,15,5,256,512,1.0
5,lstm,multitask,0.0005,0.0001,60,12,5,256,512,1.0


In [8]:
def make_cfg(family: str, task: str, run_name: str) -> TrainConfig:
    spec = dict(FINAL_TUNED_CFGS[family][task])
    # Keep only fields accepted by TrainConfig.
    train_spec = {k: spec.get(k) for k in TRAIN_KEYS if k in spec}
    train_spec["run_name"] = run_name
    return TrainConfig(**train_spec)

def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def result_row(name, model, result):
    row = {
        "model": name,
        "params": count_trainable_params(model),
        "best_epoch": result.best_epoch,
    }
    row.update(result.best_metrics)
    return row


## Train CNN baselines with imported tuned settings

In [9]:
cnn_joint_model = CNNJointModel(in_dim=in_dim)
cnn_joint_metrics = train_from_tensors(
    model=cnn_joint_model,
    X_train=bundle.X_train,
    y_train=joint_y_train,
    X_val=bundle.X_val,
    y_val=joint_y_val,
    device=device,
    cfg=make_cfg("cnn", "joint", "cnn_joint_baseline"),
)

cnn_multitask_model = CNNMultiTaskModel(in_dim=in_dim)
cnn_multitask_metrics = train_from_tensors(
    model=cnn_multitask_model,
    X_train=bundle.X_train,
    y_train=mt_y_train,
    X_val=bundle.X_val,
    y_val=mt_y_val,
    device=device,
    cfg=make_cfg("cnn", "multitask", "cnn_multitask_baseline"),
)

cnn_coordinate_model = CNNCoordinateModel(
    in_dim=in_dim,
    coordinate_std=bundle.coordinate_std,
)
cnn_coordinate_metrics = train_from_tensors(
    model=cnn_coordinate_model,
    X_train=bundle.X_train,
    y_train=coord_y_train,
    X_val=bundle.X_val,
    y_val=coord_y_val,
    device=device,
    cfg=make_cfg("cnn", "coordinate", "cnn_coordinate_baseline"),
)


epoch=001 train_loss=1.3884 val_loss=1.7724 score=0.3402
epoch=005 train_loss=0.2089 val_loss=0.8837 score=0.7246
epoch=010 train_loss=0.1005 val_loss=1.0695 score=0.7210
epoch=015 train_loss=0.0398 val_loss=0.8266 score=0.7993
epoch=020 train_loss=0.0240 val_loss=0.8149 score=0.8155
epoch=025 train_loss=0.0223 val_loss=0.9135 score=0.8110
epoch=030 train_loss=0.0159 val_loss=0.9383 score=0.8191
epoch=001 train_loss=1.4906 val_loss=2.0048 score=0.2394
epoch=005 train_loss=0.3117 val_loss=2.3150 score=0.4392
epoch=010 train_loss=0.1429 val_loss=0.9108 score=0.7732
epoch=015 train_loss=0.0887 val_loss=1.4061 score=0.6832
epoch=020 train_loss=0.0510 val_loss=1.1348 score=0.7723
epoch=025 train_loss=0.0337 val_loss=1.0524 score=0.8074
epoch=030 train_loss=0.0294 val_loss=1.1956 score=0.7804
epoch=035 train_loss=0.0223 val_loss=1.1275 score=0.8002
epoch=001 train_loss=0.5337 val_loss=0.8518 score=-115.2104
epoch=005 train_loss=0.2230 val_loss=1.3233 score=-124.8977
epoch=010 train_loss=0.16

## Train LSTM baselines with imported tuned settings

In [10]:
lstm_joint_model = LSTMJointModel(in_dim=in_dim)
lstm_joint_metrics = train_from_tensors(
    model=lstm_joint_model,
    X_train=bundle.X_train,
    y_train=joint_y_train,
    X_val=bundle.X_val,
    y_val=joint_y_val,
    device=device,
    cfg=make_cfg("lstm", "joint", "lstm_joint_baseline"),
)

lstm_multitask_model = LSTMMultiTaskModel(in_dim=in_dim)
lstm_multitask_metrics = train_from_tensors(
    model=lstm_multitask_model,
    X_train=bundle.X_train,
    y_train=mt_y_train,
    X_val=bundle.X_val,
    y_val=mt_y_val,
    device=device,
    cfg=make_cfg("lstm", "multitask", "lstm_multitask_baseline"),
)

lstm_coordinate_model = LSTMCoordinateModel(
    in_dim=in_dim,
    coordinate_std=bundle.coordinate_std,
)
lstm_coordinate_metrics = train_from_tensors(
    model=lstm_coordinate_model,
    X_train=bundle.X_train,
    y_train=coord_y_train,
    X_val=bundle.X_val,
    y_val=coord_y_val,
    device=device,
    cfg=make_cfg("lstm", "coordinate", "lstm_coordinate_baseline"),
)


epoch=001 train_loss=2.5116 val_loss=2.4578 score=0.0306
epoch=005 train_loss=0.1177 val_loss=0.3886 score=0.9163
epoch=010 train_loss=0.0397 val_loss=0.4216 score=0.9154
epoch=015 train_loss=0.0214 val_loss=0.3981 score=0.9334
epoch=020 train_loss=0.0125 val_loss=0.3763 score=0.9469
epoch=025 train_loss=0.0107 val_loss=0.4145 score=0.9433
epoch=030 train_loss=0.0068 val_loss=0.4128 score=0.9451
epoch=035 train_loss=0.0058 val_loss=0.4339 score=0.9433
epoch=001 train_loss=2.5610 val_loss=2.1515 score=0.2394
epoch=005 train_loss=0.1002 val_loss=0.4032 score=0.9109
epoch=010 train_loss=0.0416 val_loss=0.5627 score=0.9055
epoch=015 train_loss=0.0201 val_loss=0.4038 score=0.9280
epoch=020 train_loss=0.0166 val_loss=0.4394 score=0.9262
epoch=025 train_loss=0.0114 val_loss=0.4411 score=0.9370
epoch=030 train_loss=0.0101 val_loss=0.4180 score=0.9406
epoch=001 train_loss=0.9214 val_loss=0.3126 score=-59.4153
epoch=005 train_loss=0.0405 val_loss=0.0308 score=-18.1193
epoch=010 train_loss=0.0281

## Final comparison table

In [11]:
cnn_lstm_comparison_df = pd.DataFrame([
    result_row("cnn_joint", cnn_joint_model, cnn_joint_metrics),
    result_row("cnn_multitask", cnn_multitask_model, cnn_multitask_metrics),
    result_row("cnn_coordinate", cnn_coordinate_model, cnn_coordinate_metrics),
    result_row("lstm_joint", lstm_joint_model, lstm_joint_metrics),
    result_row("lstm_multitask", lstm_multitask_model, lstm_multitask_metrics),
    result_row("lstm_coordinate", lstm_coordinate_model, lstm_coordinate_metrics),
])
cnn_lstm_comparison_df


,model,params,best_epoch,epoch,train_loss,val_loss,score,joint_accuracy,building_accuracy,floor_accuracy,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m
0,cnn_joint,1584909,22,22.0,0.022929,0.826921,0.832583,0.832583,0.944194,0.842484,NaN,NaN,NaN,NaN
1,cnn_multitask,1583624,25,25.0,0.033677,1.052368,0.807381,0.807381,0.971197,0.818182,NaN,NaN,NaN,NaN
2,cnn_coordinate,1582082,56,56.0,0.050394,0.192431,-42.952327,NaN,NaN,NaN,0.449747,0.618230,42.952327,60.193217
3,lstm_joint,1642030,22,22.0,0.011822,0.370300,0.947795,0.947795,0.995500,0.947795,NaN,NaN,NaN,NaN
4,lstm_multitask,1640745,18,18.0,0.016546,0.377392,0.941494,0.941494,0.990099,0.950495,NaN,NaN,NaN,NaN
5,lstm_coordinate,1639203,43,43.0,0.018051,0.015807,-11.399431,NaN,NaN,NaN,0.128587,0.177738,11.399431,15.571116


## Parameter-count report

In [12]:
cnn_lstm_param_df = pd.DataFrame([
    {"model": "mlp_joint_reference", "params": count_trainable_params(JointMLPModel(in_dim=in_dim))},
    {"model": "mlp_multitask_reference", "params": count_trainable_params(MultiTaskMLPModel(in_dim=in_dim))},
    {"model": "mlp_coordinate_reference", "params": count_trainable_params(CoordinateMLPModel(in_dim=in_dim, coordinate_std=bundle.coordinate_std))},
    {"model": "cnn_joint", "params": count_trainable_params(cnn_joint_model)},
    {"model": "cnn_multitask", "params": count_trainable_params(cnn_multitask_model)},
    {"model": "cnn_coordinate", "params": count_trainable_params(cnn_coordinate_model)},
    {"model": "lstm_joint", "params": count_trainable_params(lstm_joint_model)},
    {"model": "lstm_multitask", "params": count_trainable_params(lstm_multitask_model)},
    {"model": "lstm_coordinate", "params": count_trainable_params(lstm_coordinate_model)},
])
cnn_lstm_param_df["params_millions"] = cnn_lstm_param_df["params"] / 1_000_000
cnn_lstm_param_df


,model,params,params_millions
0,mlp_joint_reference,1729037,1.729037
1,mlp_multitask_reference,1727752,1.727752
2,mlp_coordinate_reference,1726210,1.726210
3,cnn_joint,1584909,1.584909
4,cnn_multitask,1583624,1.583624
5,cnn_coordinate,1582082,1.582082
6,lstm_joint,1642030,1.642030
7,lstm_multitask,1640745,1.640745
8,lstm_coordinate,1639203,1.639203


## Latency benchmark

In [13]:
import time
import numpy as np
import pandas as pd
import torch
from dataclasses import dataclass, asdict

@dataclass(frozen=True)
class LatencyConfig:
    device: str = "cpu"
    batch_size: int = 1
    n_samples: int = 512
    n_warmup: int = 50
    n_repeats: int = 3
    seed: int = 42
    percentile: float = 95.0

def count_trainable_params(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def _synchronize_if_needed(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)

def _select_shared_subset(X, n_samples: int, seed: int) -> np.ndarray:
    n_total = int(len(X))
    if n_total == 0:
        raise ValueError("X is empty.")
    n_use = min(int(n_samples), n_total)
    rng = np.random.default_rng(seed)
    return rng.choice(n_total, size=n_use, replace=False)

def benchmark_single_model_latency(model: torch.nn.Module, X, cfg: LatencyConfig) -> dict[str, float]:
    if cfg.batch_size != 1:
        raise ValueError("This helper is designed for batch_size=1 fair-comparison latency.")

    device = torch.device(cfg.device)
    model = model.to(device)
    model.eval()

    X_t = torch.as_tensor(X, dtype=torch.float32)
    subset_idx = _select_shared_subset(X_t, cfg.n_samples, cfg.seed)
    X_subset = X_t[subset_idx]

    with torch.inference_mode():
        for i in range(min(cfg.n_warmup, len(X_subset))):
            xb = X_subset[i:i+1].to(device, non_blocking=False)
            _ = model(xb)
        _synchronize_if_needed(device)

    per_sample_times_ms = []
    with torch.inference_mode():
        for _rep in range(cfg.n_repeats):
            for i in range(len(X_subset)):
                xb = X_subset[i:i+1].to(device, non_blocking=False)
                _synchronize_if_needed(device)
                t0 = time.perf_counter()
                _ = model(xb)
                _synchronize_if_needed(device)
                t1 = time.perf_counter()
                per_sample_times_ms.append((t1 - t0) * 1000.0)

    arr = np.asarray(per_sample_times_ms, dtype=np.float64)
    return {
        "param_count": int(count_trainable_params(model)),
        "n_samples": int(len(X_subset)),
        "n_runs": int(len(arr)),
        "mean_ms": float(arr.mean()),
        "median_ms": float(np.median(arr)),
        "p95_ms": float(np.percentile(arr, cfg.percentile)),
        "std_ms": float(arr.std(ddof=0)),
        "min_ms": float(arr.min()),
        "max_ms": float(arr.max()),
        **{f"latency_cfg_{k}": v for k, v in asdict(cfg).items()},
    }

def benchmark_model_dict(model_dict: dict[str, torch.nn.Module], X, cfg: LatencyConfig) -> pd.DataFrame:
    rows = []
    for name, model in model_dict.items():
        print(f"Benchmarking {name}...")
        row = {"model": name}
        row.update(benchmark_single_model_latency(model, X, cfg))
        rows.append(row)
    return pd.DataFrame(rows)


In [14]:
latency_cfg = LatencyConfig(
    device="cpu",
    batch_size=1,
    n_samples=512,
    n_warmup=50,
    n_repeats=3,
    seed=42,
    percentile=95.0,
)

cnn_lstm_latency_df = benchmark_model_dict(
    model_dict={
        "cnn_joint": cnn_joint_model,
        "cnn_multitask": cnn_multitask_model,
        "cnn_coordinate": cnn_coordinate_model,
        "lstm_joint": lstm_joint_model,
        "lstm_multitask": lstm_multitask_model,
        "lstm_coordinate": lstm_coordinate_model,
    },
    X=bundle.X_val,
    cfg=latency_cfg,
)

cnn_lstm_latency_df


Benchmarking cnn_joint...
Benchmarking cnn_multitask...
Benchmarking cnn_coordinate...
Benchmarking lstm_joint...
Benchmarking lstm_multitask...
Benchmarking lstm_coordinate...


,model,param_count,n_samples,n_runs,mean_ms,median_ms,p95_ms,std_ms,min_ms,max_ms,latency_cfg_device,latency_cfg_batch_size,latency_cfg_n_samples,latency_cfg_n_warmup,latency_cfg_n_repeats,latency_cfg_seed,latency_cfg_percentile
0,cnn_joint,1584909,512,1536,1.183093,1.039759,1.096708,3.448008,0.950865,107.392188,cpu,1,512,50,3,42,95.0
1,cnn_multitask,1583624,512,1536,1.234744,1.004572,1.096841,4.458136,0.955689,131.075971,cpu,1,512,50,3,42,95.0
2,cnn_coordinate,1582082,512,1536,1.157828,0.977530,1.023014,3.223700,0.936058,92.468650,cpu,1,512,50,3,42,95.0
3,lstm_joint,1642030,512,1536,72.919812,58.489821,181.793520,56.647084,13.422549,575.005994,cpu,1,512,50,3,42,95.0
4,lstm_multitask,1640745,512,1536,63.293955,49.475772,144.929720,46.045104,10.884895,584.049309,cpu,1,512,50,3,42,95.0
5,lstm_coordinate,1639203,512,1536,60.786100,49.923708,132.249602,43.522741,10.859532,678.323700,cpu,1,512,50,3,42,95.0


## Save outputs

In [15]:
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "logs" / "fair_baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cnn_lstm_comparison_df.to_csv(OUTPUT_DIR / "cnn_lstm_baseline_results.csv", index=False)
cnn_lstm_param_df.to_csv(OUTPUT_DIR / "cnn_lstm_parameter_report.csv", index=False)
cnn_lstm_latency_df.to_csv(OUTPUT_DIR / "cnn_lstm_latency_results.csv", index=False)

print("Saved baseline outputs to:", OUTPUT_DIR)


Saved baseline outputs to: /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project/notebooks/logs/fair_baseline


## Save checkpoints

In [16]:
from pathlib import Path
import torch

# Save under project-level models/ directory
MODEL_DIR = Path("../models") if Path.cwd().name == "notebooks" else Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving checkpoints to: {MODEL_DIR.resolve()}")

def tensor_to_list(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().tolist()
    return x

def maybe_to_dict(x):
    if x is None:
        return {}
    if isinstance(x, dict):
        return x
    if hasattr(x, "__dict__"):
        return dict(x.__dict__)
    return {"value": x}

def safe_best_metrics(result_obj):
    return maybe_to_dict(getattr(result_obj, "best_metrics", {}))

def safe_best_epoch(result_obj):
    return getattr(result_obj, "best_epoch", None)

def save_checkpoint(
    path,
    model,
    model_class_name,
    task,
    in_dim,
    model_kwargs=None,
    train_cfg=None,
    best_metrics=None,
    best_epoch=None,
):
    payload = {
        "model_class_name": model_class_name,
        "task": task,
        "in_dim": in_dim,
        "model_kwargs": model_kwargs or {},
        "train_cfg": train_cfg or {},
        "best_metrics": best_metrics or {},
        "best_epoch": best_epoch,
        "state_dict": model.state_dict(),
    }
    torch.save(payload, path)
    print(f"Saved checkpoint: {path}")

Saving checkpoints to: /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project/models


In [17]:
# -----------------------------
# CNN checkpoints
# -----------------------------
save_checkpoint(
    MODEL_DIR / "cnn_joint.pt",
    cnn_joint_model,
    model_class_name=type(cnn_joint_model).__name__,
    task="joint",
    in_dim=in_dim,
    model_kwargs={},
    best_metrics=safe_best_metrics(cnn_joint_metrics),
    best_epoch=safe_best_epoch(cnn_joint_metrics),
)

save_checkpoint(
    MODEL_DIR / "cnn_multitask.pt",
    cnn_multitask_model,
    model_class_name=type(cnn_multitask_model).__name__,
    task="multitask",
    in_dim=in_dim,
    model_kwargs={},
    best_metrics=safe_best_metrics(cnn_multitask_metrics),
    best_epoch=safe_best_epoch(cnn_multitask_metrics),
)

save_checkpoint(
    MODEL_DIR / "cnn_coordinate.pt",
    cnn_coordinate_model,
    model_class_name=type(cnn_coordinate_model).__name__,
    task="coordinate",
    in_dim=in_dim,
    model_kwargs={
        "coordinate_std": tensor_to_list(bundle.coordinate_std),
    },
    best_metrics=safe_best_metrics(cnn_coordinate_metrics),
    best_epoch=safe_best_epoch(cnn_coordinate_metrics),
)

# -----------------------------
# LSTM checkpoints
# -----------------------------
save_checkpoint(
    MODEL_DIR / "lstm_joint.pt",
    lstm_joint_model,
    model_class_name=type(lstm_joint_model).__name__,
    task="joint",
    in_dim=in_dim,
    model_kwargs={},
    best_metrics=safe_best_metrics(lstm_joint_metrics),
    best_epoch=safe_best_epoch(lstm_joint_metrics),
)

save_checkpoint(
    MODEL_DIR / "lstm_multitask.pt",
    lstm_multitask_model,
    model_class_name=type(lstm_multitask_model).__name__,
    task="multitask",
    in_dim=in_dim,
    model_kwargs={},
    best_metrics=safe_best_metrics(lstm_multitask_metrics),
    best_epoch=safe_best_epoch(lstm_multitask_metrics),
)

save_checkpoint(
    MODEL_DIR / "lstm_coordinate.pt",
    lstm_coordinate_model,
    model_class_name=type(lstm_coordinate_model).__name__,
    task="coordinate",
    in_dim=in_dim,
    model_kwargs={
        "coordinate_std": tensor_to_list(bundle.coordinate_std),
    },
    best_metrics=safe_best_metrics(lstm_coordinate_metrics),
    best_epoch=safe_best_epoch(lstm_coordinate_metrics),
)

Saved checkpoint: ../models/cnn_joint.pt
Saved checkpoint: ../models/cnn_multitask.pt
Saved checkpoint: ../models/cnn_coordinate.pt
Saved checkpoint: ../models/lstm_joint.pt
Saved checkpoint: ../models/lstm_multitask.pt
Saved checkpoint: ../models/lstm_coordinate.pt
